# Step 33 — Person-level mode choice: car versus transit on the V2 areas

Step 31 (`Mode_skims_and_flow_comparison.ipynb`, METHODOLOGY §6ac) tried to read the cost
sensitivity λ off the 2022 cross-section of area pairs and failed: the trip-weighted binary
logit of the transit share on `GC_bus − GC_car` returned the wrong sign (λ = −0.011 per
generalized minute), because the pairs where the bus is dearest relative to the car are also
the least car-available. Car availability is not in the skims, so at the pair level it
dominates the cross-section and masks any genuine cost response. Task E7 of
`docs/CORRIDOR_DEMAND_TASKS.md` and item 3a.1 of `docs/PLAN_TIGHTENING_AND_SCENARIOS.md`
name the remedy: estimate at the **person level**, where car availability, licence, purpose,
age and sector are held constant, and only then read λ.

**This notebook** builds that estimation sample from the THS 2017/18 trips file joined to the
person and household tables (`Input/THS_2017-2018/`, pulled 23 September 2026 with the field
metadata), attaches the step 31 skims per area pair, and estimates a set of weighted binary
logits — car versus transit (bus + Metronit + rail; taxi-type out of the choice set as in step
31) — on the AM trips with both ends in the 25 V2 areas. **The only weight used is `new_wf`**
from the trips file (instruction of 23 September 2026); `wf3` and the person / household files'
own weights, if any, are not used.

Two things the metadata settled on the way (see §1 below): the activity codes of the trips
file (`mainActivity` 1–12) map one-to-one onto the activities file's text codes, and the mode
codes are **3 Public Bus, 4 group taxi (sherut), 5 Matronit, 7 train, 8 special taxi** — the
chain's current mapping (`BUS = [3, 4]`, `TAXI = [5, 8]`) has 4 and 5 swapped. The estimation
uses the corrected mapping; steps 15–32 were rerun with the corrected codes the same day (METHODOLOGY §8, caveat 14).

In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import norm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/mode_choice'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
THS = 'Input/THS_2017-2018'
LAMBDA_ASSUMED = 0.03            # step 31's central assumption, per generalized minute (range 0.02–0.05)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
for f in ['trips_ths_2017.xlsx', 'PersonsFin2.csv', 'HHfinal.csv', 'ACTIVITIES_DEC18_corrected.csv']:
    assert not is_pointer(f'{THS}/{f}'), f'{f} is a Git LFS pointer — pull Input/THS_2017-2018 first'

## 1. The trips file, its codes, and the person and household attributes

The trips file (`trips_ths_2017.xlsx`, 146,394 rows) holds one row per activity (place) per
person per survey day. A trip runs from the previous row's `actTaz` to the current row's; its
departure hour is the origin row's `Dep_h`, its mode the destination row's `mainmode` — the
rule of step 5 (§6c) and step 15 (§6m), reused unchanged. Persons are keyed by `HHID3` and
`IndID3`; `PersonsFin2.csv` keys the same people as `INDIVID = HHID × 100 + personid`.

**Code check against the activities file.** The activities file carries the same stops with
text codes. Joining the two on household, person, day and stop order, and keeping only the
stops whose start times agree to within two minutes (117,410 rows), gives a one-to-one mapping
for both `mainActivity` and `mainmode`. The mode result matters: code **4 is group taxi** and
code **5 is Matronit**, not the other way round.

In [3]:
tr = pd.read_excel(f'{THS}/trips_ths_2017.xlsx')
assert set(tr['SurveyDay'].unique()) == {1, 2}
act = pd.read_csv(f'{THS}/ACTIVITIES_DEC18_corrected.csv', usecols=['HHID', 'INDIVID', 'ACT_DAY', 'ACT_ID', 'mainActivity', 'MODE_NAME', 'StartTime'])
act['IndID'] = act['INDIVID'] % 100; act['day'] = act['ACT_DAY'].map({10: 1, 20: 2})
act['st'] = pd.to_datetime(act['StartTime'], dayfirst=True)
key = ['HHID', 'IndID', 'day', 'ACT_ID']
act1 = act.drop_duplicates(key, keep=False)                                       # keys that occur once
tr['st'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(tr['arrtime'], unit='D')   # Excel serial date
m = tr.merge(act1, left_on=['HHID3', 'IndID3', 'SurveyDay', 'placeno'], right_on=key, how='inner', suffixes=('', '_act'))
m = m[(m['st'] - m['st_act']).dt.total_seconds().abs() < 120]
ct_act = pd.crosstab(m['mainActivity'], m['mainActivity_act']); ct_mode = pd.crosstab(m['mainmode'], m['MODE_NAME'])
ACT_NAME = ct_act.idxmax(axis=1).to_dict(); MODE_NAME = ct_mode.idxmax(axis=1).to_dict()
purity_act = (ct_act.max(axis=1) / ct_act.sum(axis=1)); purity_mode = (ct_mode.max(axis=1) / ct_mode.sum(axis=1))
codes = pd.concat([pd.DataFrame({'field': 'mainActivity', 'code': ct_act.index, 'meaning': ct_act.idxmax(axis=1).values, 'matched rows': ct_act.sum(axis=1).values, 'share on the modal meaning': purity_act.values}),
                   pd.DataFrame({'field': 'mainmode', 'code': ct_mode.index, 'meaning': ct_mode.idxmax(axis=1).values, 'matched rows': ct_mode.sum(axis=1).values, 'share on the modal meaning': purity_mode.values})])
codes.to_csv(f'{OUT}/trips_file_code_check.csv', index=False, float_format='%.4f')
print(f'stops matched to the activities file on key and start time: {len(m):,} of {len(tr):,}')
print(codes.round(3).to_string(index=False))
assert MODE_NAME[3] == 'Public Bus' and MODE_NAME[5] == 'Matronit' and MODE_NAME[4] == 'Group Taxi' and MODE_NAME[8] == 'Special Taxi' and MODE_NAME[7] == 'Train'
assert MODE_NAME[10] == 'Vehicle as Driver' and MODE_NAME[11] == 'Vehicle as Passenger'

stops matched to the activities file on key and start time: 117,410 of 146,394
       field  code                             meaning  matched rows  share on the modal meaning
mainActivity     1                                Home         55495                       0.998
mainActivity     2                                Work         10769                       1.000
mainActivity     3               Work Related business          6718                       1.000
mainActivity     4                           Education          9903                       0.999
mainActivity     5                            Shopping          5095                       1.000
mainActivity     6             Personal Errands/Prayer          8093                       1.000
mainActivity     7 Social visit with friends or family          6685                       1.000
mainActivity     8                     Medical/Dentist          1177                       1.000
mainActivity     9             Entertainment/Mea

**Mode groups.** With the codes confirmed: **car** = 10 driver, 11 passenger; **transit** = 3
Public Bus, 5 Matronit, 7 train; **taxi-type** = 4 group taxi, 8 special taxi (out of the
choice set, as in step 31); everything else (walk, bicycle, chartered / school bus 9,
motorcycle 12, other 13, 99) is OTHER and out. The chain's mapping (`BUS = [3, 4]`,
`TAXI = [5, 8]`) would have put the Matronit rows into taxi-type and the group-taxi rows into
bus; the count of AM rows affected is printed below and carried to §8 of the methodology.

In [4]:
COMP = {'CAR': [10, 11], 'BUS': [3, 5], 'TAXI': [4, 8], 'RAIL': [7]}               # corrected mapping
COMP_OLD = {'CAR': [10, 11], 'BUS': [3, 4], 'TAXI': [5, 8], 'RAIL': [7]}           # steps 15–16 as they stand
def comp_of(mapping):
    inv = {c: k for k, v in mapping.items() for c in v}
    return lambda x: inv.get(x, 'OTHER')

d = tr.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy()
g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1); d['trip_dep_h'] = g['Dep_h'].shift(1); d['o_act'] = g['mainActivity'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h']).copy()
trips = trips[trips['trip_dep_h'].isin([6, 7, 8])].copy()
trips['comp'] = trips['mainmode'].map(comp_of(COMP)); trips['comp_old'] = trips['mainmode'].map(comp_of(COMP_OLD))
swap = trips[trips['comp'] != trips['comp_old']]
print(f'AM-peak trips (departure hour 6–8): {len(trips):,} sampled rows, {trips["new_wf"].sum() / 2:,.0f} average-weekday expanded')
print('rows whose layer changes under the corrected codes (whole study area, AM):')
print(swap.groupby(['mainmode', 'comp_old', 'comp']).agg(sampled=('new_wf', 'size'), expanded_avg_day=('new_wf', lambda s: s.sum() / 2)).round(0).to_string())

AM-peak trips (departure hour 6–8): 28,060 sampled rows, 2,184,632 average-weekday expanded
rows whose layer changes under the corrected codes (whole study area, AM):
                        sampled  expanded_avg_day
mainmode comp_old comp                           
4        BUS      TAXI       26            2006.0
5        TAXI     BUS       182           11406.0


**Person and household attributes.** From `PersonsFin2.csv`: car licence, household cars
(`HHnCars`), age, gender, employment, whether currently in education, workplace parking. From
`HHfinal.csv`: sector (`migzar`), household type, household size. Licensed drivers per household
are counted from the person table. The **car-availability segment** of a traveller is:

| segment | rule |
|---|---|
| `no_car_hh` | household has no car |
| `no_licence` | household has a car, the traveller has no licence (or is a child) |
| `competition` | licence, and fewer cars than licensed drivers in the household |
| `car_available` | licence, and at least one car per licensed driver |

**Trip purpose** is home-based on the non-home end's activity: `HBW` = work or work-related
business (codes 2, 3), `HBE` = education (4), `HBO` = any other home-based, `NHB` = neither end
home. The activity-code mapping is the one confirmed above.

In [5]:
per = pd.read_csv(f'{THS}/PersonsFin2.csv', encoding='utf-8-sig'); hh = pd.read_csv(f'{THS}/HHfinal.csv', encoding='utf-8-sig')
per['licence'] = per['license_car'].eq('yes')
lic_hh = per.groupby('HHID')['licence'].sum().rename('licensed_drivers')
per = per.merge(lic_hh, on='HHID', how='left').merge(hh[['HHID', 'migzar', 'HHSIZE', 'HHVEHICLE']], on='HHID', how='left')
def segment(r):
    if r['HHnCars'] == 0: return 'no_car_hh'
    if not r['licence']: return 'no_licence'
    return 'competition' if r['HHnCars'] < r['licensed_drivers'] else 'car_available'
per['car_seg'] = per.apply(segment, axis=1)
per['age_grp'] = pd.cut(per['Age'], [0, 17, 24, 34, 64, 200], labels=['0-17', '18-24', '25-34', '35-64', '65+'])
per['sector'] = np.where(per['migzar'].isin([4, 5]), 'arab', 'general')          # migzar 4 / 5 are the Arab-sector households (HHTYPE cross-tab below); 1–3 the rest
per['student'] = per['education'].eq('yes'); per['female'] = per['Gender'].eq('female')
per['employed'] = per['employment'].isin(['full', 'part'])
per['paid_parking'] = per['workparking'].isin(['paid', 'pay'])
print('employment codes:', per['employment'].value_counts(dropna=False).to_dict())
print('workparking codes:', per['workparking'].value_counts(dropna=False).to_dict())
print('sector (migzar) by household type:'); print(pd.crosstab(hh['migzar'], hh['HHTYPE']).to_string())

trips['INDIVID'] = trips['HHID3'] * 100 + trips['IndID3']
trips = trips.merge(per[['INDIVID', 'HHID', 'Age', 'age_grp', 'female', 'licence', 'HHnCars', 'licensed_drivers', 'car_seg', 'employed', 'student', 'paid_parking', 'sector', 'HHSIZE']], on='INDIVID', how='left')
print(f'trips with a person record: {trips["HHID"].notna().mean():.4f}')
ACT = {1: 'Home', 2: 'Work', 3: 'WorkRelated', 4: 'Education'}
def purpose(r):
    o, dd = r['o_act'], r['mainActivity']
    if o == 1 and dd == 1: return 'HBO'
    if o != 1 and dd != 1: return 'NHB'
    a = dd if o == 1 else o
    return 'HBW' if a in (2, 3) else 'HBE' if a == 4 else 'HBO'
trips['purpose'] = trips.apply(purpose, axis=1)

employment codes: {'full': 5604, nan: 4906, 'retired': 1597, 'unemployed_not': 1500, 'part': 1187, 'care': 874, 'unemployed': 441, 'service': 127, 'unknown': 109, 'army': 43}
workparking codes: {nan: 11189, 'free': 3856, 'no_need': 531, 'no_but_need': 444, 'no_prob': 214, 'unknown': 110, 'subsidized': 44}
sector (migzar) by household type:
HHTYPE  arab  orto  other  religious  secular
migzar                                       
1         76    99      0         71      980
2         46   191      3        341     1391
3          1     8      3         74      317
4       1280     0      0          3       25
5        186     0      0          0        3
trips with a person record: 0.9996


## 2. Area pairs on the V2 aggregation

`actTaz` is on the national 2636-zone system; `Input/TAZ_2636_Keys.xlsx` takes it to the
1250-zone system, whose ids are the `TAZ_1270` of the study keys, and each 1250-zone holds up
to eight study TAZs. As in step 15, the one-to-many link is split by 2020 **population** on the
origin side and **employment** on the destination side (`Input/Zonal_2020.csv`; fallback to the
other variable, then uniform). Each study TAZ then maps to a V2 area through the `TazAgg` key of
`Input/Corridor_TAZ_Agg_V2.xlsx` (174 TAZs, 25 areas). A sampled trip therefore lands on an area
pair with a **probability** — the product of its origin and destination shares — and the
estimation sample carries one row per trip × area pair with weight `new_wf × share`. Most
1250-zones sit inside one area, so the expansion is small (the table below counts it).

The sample is restricted as step 31's flow comparison was: both ends in the 25 areas, off the
diagonal (the transit skims are not filled within an area), car or transit.

In [6]:
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636'); map_2636_1250 = k26.set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(map_2636_1250); trips['d1250'] = trips['actTaz'].map(map_2636_1250)
KEYS_LFS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
keys_raw = pd.read_csv('Input/taz_keys_from_shapefile.csv') if is_pointer(KEYS_LFS) else pd.read_csv(KEYS_LFS, encoding='windows-1255')
kn = keys_raw[['TAZ_1270', 'TAZ_NUMBER', 'SZ_NEW']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int, 'SZ_NEW': int}).drop_duplicates('TAZ_NUMBER')
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID')
kn['pop'] = kn['TAZ_NUMBER'].map(zon['POPULATION']).fillna(0); kn['emp'] = kn['TAZ_NUMBER'].map(zon['EMPL_TOT']).fillna(0)
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); tazagg = xl.parse('TazAgg').set_index('TAZ')['AggCode']
AREAS = list(areas.index); NAME = areas['AggAreaName']
kn['area'] = kn['TAZ_NUMBER'].map(tazagg)

def area_shares(primary, secondary):
    """1250-zone -> {area: share}: the split of step 15 collapsed to the V2 areas (TAZs outside the areas keep their share under 'none')."""
    out = {}
    for z, grp in kn.groupby('TAZ_1270'):
        for v in (grp[primary].values, grp[secondary].values, np.ones(len(grp))):
            if v.sum() > 0: break
        s = pd.Series(v / v.sum(), index=grp['area'].fillna(-1).values).groupby(level=0).sum()
        out[z] = s.to_dict()
    return out
SH_O, SH_D = area_shares('pop', 'emp'), area_shares('emp', 'pop')

base = trips[trips['comp'].isin(['CAR', 'BUS', 'RAIL'])].copy()
base['transit'] = base['comp'].isin(['BUS', 'RAIL']).astype(int)
rows = []
for r in base.itertuples(index=False):
    so, sd = SH_O.get(r.o1250), SH_D.get(r.d1250)
    if not so or not sd: continue
    for ao, po in so.items():
        if ao == -1: continue
        for ad, pd_ in sd.items():
            if ad == -1 or ad == ao: continue
            rows.append((r.PerID3, r.HHID, r.SurveyDay, r.placeno, ao, ad, po * pd_, r.new_wf * po * pd_))
S = pd.DataFrame(rows, columns=['PerID3', 'HHID', 'SurveyDay', 'placeno', 'o', 'd', 'share', 'w'])
S = S.merge(base[['PerID3', 'SurveyDay', 'placeno', 'comp', 'transit', 'new_wf', 'Age', 'age_grp', 'female', 'licence', 'HHnCars', 'licensed_drivers', 'car_seg', 'employed', 'student', 'paid_parking', 'sector', 'HHSIZE', 'purpose', 'mainmode']], on=['PerID3', 'SurveyDay', 'placeno'], how='left')
S['o'] = S['o'].astype(int); S['d'] = S['d'].astype(int)
n_trips = S.groupby(['PerID3', 'SurveyDay', 'placeno']).ngroups
print(f'sampled car / transit AM trips with both ends in the 25 areas, off-diagonal: {n_trips:,} trips -> {len(S):,} trip × pair rows '
      f'(rows per trip {len(S) / n_trips:.2f}); expanded {S["w"].sum() / 2:,.0f} per average weekday')
print(f'sum of pair shares per trip: min {S.groupby(["PerID3", "SurveyDay", "placeno"])["share"].sum().min():.3f} (below 1 where part of a zone lies outside the areas or on the diagonal)')

sampled car / transit AM trips with both ends in the 25 areas, off-diagonal: 1,647 trips -> 2,325 trip × pair rows (rows per trip 1.41); expanded 67,470 per average weekday
sum of pair shares per trip: min 0.000 (below 1 where part of a zone lies outside the areas or on the diagonal)


In [7]:
# what the corrected mode codes do to the corridor-internal layers (both ends in the 25 areas, off-diagonal, all four layers): the count step 31 worked with until the chain was rerun with the corrected codes on 23 September 2026
def area_pair_weight(df):
    tot = {}
    for r in df.itertuples(index=False):
        so, sd = SH_O.get(r.o1250), SH_D.get(r.d1250)
        if not so or not sd: continue
        w = sum(po * pd_ for ao, po in so.items() if ao != -1 for ad, pd_ in sd.items() if ad != -1 and ad != ao)
        tot[r.comp] = tot.get(r.comp, 0) + r.new_wf * w / 2
    return pd.Series(tot)
layers = pd.DataFrame({'corrected codes (3, 5 bus; 4, 8 taxi)': area_pair_weight(trips.assign(comp=trips['comp'])), 'codes as read before 23 Sep 2026 (3, 4 bus; 5, 8 taxi)': area_pair_weight(trips.assign(comp=trips['comp_old']))}).reindex(['CAR', 'BUS', 'TAXI', 'RAIL', 'OTHER']).round(0)
layers['change'] = layers.iloc[:, 0] - layers.iloc[:, 1]
layers.to_csv(f'{OUT}/v2_area_layers_by_mode_code_mapping.csv'); print('average-weekday AM trips within the 25 V2 areas, off-diagonal, survey expansion (2018 level):'); print(layers.to_string())

average-weekday AM trips within the 25 V2 areas, off-diagonal, survey expansion (2018 level):
       corrected codes (3, 5 bus; 4, 8 taxi)  codes as read before 23 Sep 2026 (3, 4 bus; 5, 8 taxi)  change
CAR                                  54732.0                                                 54732.0     0.0
BUS                                  12673.0                                                  6747.0  5926.0
TAXI                                   723.0                                                  6650.0 -5927.0
RAIL                                    64.0                                                    64.0     0.0
OTHER                                10655.0                                                 10655.0     0.0


## 3. The skims per pair

From step 31 (`Output/skims/`): the car and bus generalized cost (`GC = IVT + 2·walk + 2·wait +
8·transfers`, money out by decision), the Metronit GC where a direct service exists, and the
centroid distance. `dGC = GC_bus − GC_car` is the variable step 31 used; `dGC_best` takes the
cheaper of bus and Metronit as the transit cost. Distance bands are step 31's.

In [8]:
def skim(name):
    m = pd.read_csv(f'Output/skims/skim_{name}.csv', index_col=0).rename(columns=int)
    return pd.Series(m.to_numpy().ravel(), index=pd.MultiIndex.from_product([m.index, m.columns]), name=name)
sk = pd.concat([skim('car_gc'), skim('bus_gc'), skim('brt_gc'), skim('bus_ivt'), skim('bus_walk'), skim('bus_wait'), skim('bus_transfers'), skim('car_ivt')], axis=1)
sk.index.names = ['o', 'd']; sk = sk.reset_index()
sk['tr_best_gc'] = sk[['bus_gc', 'brt_gc']].min(axis=1); sk['has_brt'] = sk['brt_gc'].notna().astype(int)
sk['dGC'] = sk['bus_gc'] - sk['car_gc']; sk['dGC_best'] = sk['tr_best_gc'] - sk['car_gc']
ac = pd.read_csv('Output/lrt_v2/lrt_area_representative_station.csv', index_col=0).reindex(AREAS)
sk['dist_km'] = [np.hypot(ac.loc[o, 'cx'] - ac.loc[dd, 'cx'], ac.loc[o, 'cy'] - ac.loc[dd, 'cy']) / 1000 for o, dd in zip(sk['o'], sk['d'])]
sk['band'] = pd.cut(sk['dist_km'], [0, 3, 6, 10, 20, 100], right=False, labels=['<3', '3-6', '6-10', '10-20', '20+'])
S = S.merge(sk, on=['o', 'd'], how='left')
S = S[S['dGC'].notna()].copy()
S['w_norm'] = S['w'] * len(S) / S['w'].sum()                # new_wf-based, rescaled to the row count so standard errors reflect the sample
S['transit_lbl'] = S['transit'].map({0: 'car', 1: 'transit'})
S.to_csv(f'{OUT}/estimation_sample_person_level.csv', index=False, float_format='%.5f')

summ = S.groupby('car_seg').apply(lambda g: pd.Series({'trip rows': len(g), 'expanded (avg day)': g['w'].sum() / 2, 'transit share (weighted)': np.average(g['transit'], weights=g['w']), 'transit share (unweighted)': g['transit'].mean(), 'mean dGC (weighted)': np.average(g['dGC'], weights=g['w'])}))
summ.loc['all'] = [len(S), S['w'].sum() / 2, np.average(S['transit'], weights=S['w']), S['transit'].mean(), np.average(S['dGC'], weights=S['w'])]
summ.to_csv(f'{OUT}/sample_by_car_segment.csv', float_format='%.4f'); print(summ.round(3).to_string())
byp = S.groupby('purpose').apply(lambda g: pd.Series({'trip rows': len(g), 'transit share (weighted)': np.average(g['transit'], weights=g['w'])}))
print(); print(byp.round(3).to_string())
print(); print('transit rows by mode code:', S[S['transit'] == 1].groupby('mainmode')['w'].size().to_dict())

               trip rows  expanded (avg day)  transit share (weighted)  transit share (unweighted)  mean dGC (weighted)
car_seg                                                                                                                
car_available      807.0           22525.861                     0.019                       0.021               18.423
competition        636.0           20334.127                     0.109                       0.083               18.503
no_car_hh          319.0           10282.628                     0.689                       0.592               16.295
no_licence         556.0           13872.484                     0.217                       0.169               11.155
all               2319.0           67058.704                     0.190                       0.152               16.608



         trip rows  transit share (weighted)
purpose                                     
HBE          332.0                     0.294
HBO          714.0                     0.100
HBW          593.0                     0.288
NHB          680.0                     0.101

transit rows by mode code: {3: 220, 5: 126, 7: 7}


## 4. Estimation

Weighted binary logits, weight `new_wf` (rescaled to the row count), standard errors clustered
by household (the two survey days of one household are not independent observations). The
sequence reproduces the identification problem and then removes it:

- **M0** — `dGC` only: the person-level counterpart of step 31's pair-level fit. Same
  confound, so the same wrong sign is expected.
- **M1** — `dGC` plus the car-availability segment, purpose, age, gender, sector, student and
  distance band. λ is the coefficient on `dGC` with the composition held constant.
- **M2** — M1 with λ allowed to differ by car-availability segment.
- **M3** — M1 with origin- and destination-area fixed effects: λ identified only from the
  pair-specific cost, with everything about the areas themselves absorbed.
- **M1b / M2b** — M1 and M2 with `dGC_best` (Metronit where it runs).
- **M1u** — M1 unweighted, as a check on the weights.
- **M4** — M1 on the licence holders in car-owning households only (`car_available` and
  `competition`): the travellers who actually choose between car and transit, and the segment
  step 31's capture draws from the car.

The model's λ is the negative of the coefficient on `dGC` (a higher transit cost lowers the
transit share). It is comparable with step 31's assumed λ = 0.03 per generalized minute.

In [9]:
class Fit:
    """Weighted binary logit by IRLS (statsmodels GLM, var_weights = the new_wf-based weight) with household-clustered sandwich standard errors computed directly."""
    def __init__(self, formula, data, weighted=True, label=''):
        data = data[data['HHID'].notna()]
        data = data.loc[smf.glm(formula, data=data, family=sm.families.Binomial()).data.row_labels]   # rows the formula keeps (no missing covariate)
        w = data['w_norm'].to_numpy() if weighted else np.ones(len(data))
        mdl = smf.glm(formula, data=data, family=sm.families.Binomial(), var_weights=w)
        res = mdl.fit(maxiter=200)
        X, y = mdl.exog, mdl.endog; p = res.predict()
        bread = np.linalg.pinv(X.T @ (X * (w * p * (1 - p))[:, None]))
        score = X * (w * (y - p))[:, None]
        grp = pd.factorize(data['HHID'])[0]; G = grp.max() + 1
        sg = np.zeros((G, X.shape[1])); np.add.at(sg, grp, score)
        V = bread @ (sg.T @ sg) @ bread * G / (G - 1)
        self.params = pd.Series(res.params.values, index=mdl.exog_names); self.bse = pd.Series(np.sqrt(np.diag(V)), index=mdl.exog_names)
        z = self.params / self.bse; self.pvalues = pd.Series(2 * (1 - norm.cdf(np.abs(z.values))), index=mdl.exog_names)
        null = smf.glm('transit ~ 1', data=data, family=sm.families.Binomial(), var_weights=w).fit()
        self.llf, self.rho2, self.label, self.n, self.clusters = res.llf, 1 - res.llf / null.llf, label, len(data), G
        self.table = pd.DataFrame({'coef': self.params, 'se': self.bse, 'z': z, 'p': self.pvalues})
def fit(*a, **k): return Fit(*a, **k)

CTRL = ' + C(car_seg, Treatment("car_available")) + C(purpose, Treatment("HBW")) + C(age_grp, Treatment("35-64")) + female + student + C(sector, Treatment("general")) + C(band, Treatment("6-10"))'
M = {}
M['M0'] = fit('transit ~ dGC', S, label='M0: dGC only')
M['M1'] = fit('transit ~ dGC' + CTRL, S, label='M1: dGC + car availability, purpose, person, distance band')
M['M2'] = fit('transit ~ dGC:C(car_seg)' + CTRL, S, label='M2: λ by car-availability segment')
M['M3'] = fit('transit ~ dGC' + CTRL + ' + C(o) + C(d)', S, label='M3: M1 + origin and destination area fixed effects')
M['M1b'] = fit('transit ~ dGC_best' + CTRL, S, label='M1b: M1 with the cheaper of bus and Metronit')
M['M2b'] = fit('transit ~ dGC_best:C(car_seg)' + CTRL, S, label='M2b: M2 with the cheaper of bus and Metronit')
M['M1u'] = fit('transit ~ dGC' + CTRL, S, weighted=False, label='M1u: M1 unweighted')
CHOICE = S[S['car_seg'].isin(['car_available', 'competition'])]
M['M4'] = fit('transit ~ dGC + C(car_seg, Treatment("car_available")) + C(purpose, Treatment("HBW")) + C(age_grp, Treatment("35-64")) + female + student + C(sector, Treatment("general")) + C(band, Treatment("6-10"))', CHOICE, label='M4: licence holders in car-owning households only (the segment the LRT would draw from the car)')

def lam_rows(res):
    out = []
    for name in res.params.index:
        if name.startswith('dGC'):
            seg = name.split('[')[-1].rstrip(']') if ':' in name else 'all'
            out.append({'model': res.label, 'segment': seg, 'lambda (per gen-min)': -res.params[name], 'se': res.bse[name], 'p': res.pvalues[name]})
    return out
lam = pd.DataFrame([r for res in M.values() for r in lam_rows(res)])
lam['rho2'] = lam['model'].map({res.label: res.rho2 for res in M.values()}); lam['rows'] = lam['model'].map({res.label: res.n for res in M.values()})
lam['ratio to assumed 0.03'] = lam['lambda (per gen-min)'] / LAMBDA_ASSUMED
lam.to_csv(f'{OUT}/lambda_estimates.csv', index=False, float_format='%.5f')
print(lam.round(4).to_string(index=False))

                                                                                          model       segment  lambda (per gen-min)     se      p   rho2  rows  ratio to assumed 0.03
                                                                                   M0: dGC only           all                0.0126 0.0115 0.2712 0.0031  2318                 0.4206
                                     M1: dGC + car availability, purpose, person, distance band           all                0.0350 0.0167 0.0363 0.4325  2310                 1.1651
                                                              M2: λ by car-availability segment car_available                0.0526 0.0296 0.0756 0.4438  2310                 1.7546
                                                              M2: λ by car-availability segment   competition               -0.0064 0.0212 0.7631 0.4438  2310                -0.2133
                                                              M2: λ by car-availability se

In [10]:
coef = pd.concat([pd.DataFrame({'model': res.label, 'term': res.params.index, 'coef': res.params.values, 'se': res.bse.values, 'p': res.pvalues.values}) for res in M.values()])
coef = coef[~coef['term'].str.startswith('C(o)') & ~coef['term'].str.startswith('C(d)')]
coef.to_csv(f'{OUT}/coefficients_all_models.csv', index=False, float_format='%.5f')
print(M['M1'].table.round(4).to_string())

                                                         coef      se       z       p
Intercept                                             -2.7468  0.6145 -4.4702  0.0000
C(car_seg, Treatment("car_available"))[T.competition]  2.1385  0.5736  3.7283  0.0002
C(car_seg, Treatment("car_available"))[T.no_car_hh]    5.5402  0.6101  9.0808  0.0000
C(car_seg, Treatment("car_available"))[T.no_licence]   4.1686  0.6444  6.4691  0.0000
C(purpose, Treatment("HBW"))[T.HBE]                    0.4159  0.6022  0.6907  0.4897
C(purpose, Treatment("HBW"))[T.HBO]                   -2.0176  0.5805 -3.4755  0.0005
C(purpose, Treatment("HBW"))[T.NHB]                   -1.6200  0.5856 -2.7663  0.0057
C(age_grp, Treatment("35-64"))[T.0-17]                -2.1995  0.5679 -3.8733  0.0001
C(age_grp, Treatment("35-64"))[T.18-24]                0.4016  0.4508  0.8909  0.3730
C(age_grp, Treatment("35-64"))[T.25-34]               -0.0059  0.6095 -0.0096  0.9923
C(age_grp, Treatment("35-64"))[T.65+]                 

## 5. Reading the result

In [11]:
# transit share against dGC, by car-availability segment: trip-weighted means in cost-difference bins, with M2's fitted curves at the segment's mean covariates
bins = [-20, 0, 5, 10, 15, 20, 30, 45, 90]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
SEGS = ['no_car_hh', 'no_licence', 'competition', 'car_available']; COL = dict(zip(SEGS, [PURPLE, ORANGE, AQUA, BLUE]))
for ax, col in zip(axes, ['dGC', 'dGC_best']):
    for seg in SEGS:
        g = S[S['car_seg'] == seg].copy(); g['bin'] = pd.cut(g[col], bins)
        b = pd.DataFrame([{'mid': k.mid, 'share': np.average(x['transit'], weights=x['w'])} for k, x in g.groupby('bin', observed=True)])
        ax.plot(b['mid'], b['share'], 'o-', color=COL[seg], label=f'{seg} (observed, binned)', ms=5)
    ax.set_xlabel(f'{col}: transit GC − car GC (generalized minutes)'); ax.grid(color=GRID, lw=0.6); ax.set_ylim(0, 1)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
axes[0].set_ylabel('transit share of car + transit (new_wf-weighted)'); axes[0].legend(frameon=False, fontsize=8)
axes[0].set_title('bus skim', loc='left', fontsize=10); axes[1].set_title('cheaper of bus and Metronit', loc='left', fontsize=10)
fig.suptitle('Observed transit share against the cost difference, by car-availability segment (AM trips, 25 V2 areas, THS 2017/18)', fontsize=11)
fig.savefig('Output/figures/mode_choice_person_level_by_segment.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

In [12]:
# the estimate against step 31's λ cases: point estimate and 95 % interval of M1 (the central specification), M3 and M4 beside the assumed 0.03 (0.02–0.05)
cap = pd.read_csv('Output/skims/lrt_capture_scenarios.csv')
print(cap.to_string(index=False))
def gc_term(res): return [t for t in res.params.index if t.startswith('dGC')][0]
ci = pd.DataFrame([{'model': M[k].label, 'lambda': -M[k].params[gc_term(M[k])], 'lo95': -M[k].params[gc_term(M[k])] - 1.96 * M[k].bse[gc_term(M[k])], 'hi95': -M[k].params[gc_term(M[k])] + 1.96 * M[k].bse[gc_term(M[k])], 'rows': M[k].n, 'households': M[k].clusters} for k in ['M1', 'M1b', 'M3', 'M4', 'M1u']])
ci['assumed (step 31)'] = LAMBDA_ASSUMED; ci['assumed range'] = '0.02–0.05'
ci.to_csv(f'{OUT}/lambda_summary.csv', index=False, float_format='%.4f'); print(); print(ci.round(4).to_string(index=False))

           scenario                                  case    λ  λ_T  LRT premium  LRT trips 06–09  of which from bus  of which from car  bus trips after  car trips after  transit share before  transit share after  LRT share of transit  LRT trips on trunk pairs  trunk pairs P_LRT|T (trip-wtd)
LRT all underground central (λ 0.03, λ_T 0.06, premium 5) 0.03 0.06          5.0             4250               3837                413            11091            55237                0.2003               0.2174                 0.277                      1765                           0.344
LRT all underground        low λ (0.02 / 0.03, premium 5) 0.02 0.03          5.0             6028               5111                917            10519            54032                0.2003               0.2344                 0.364                      2314                           0.419
LRT all underground       high λ (0.05 / 0.10, premium 5) 0.05 0.10          5.0             3161               2886     